# LeetCode 708 — Insert into a Sorted Circular Linked List bằng Python

Link đề: https://leetcode.com/problems/insert-into-a-sorted-circular-linked-list/ · Độ khó: **Medium** · Chủ đề: Linked List

> **Ghi chú Premium:** đây là bài **Premium** (🔒). Đề trong notebook được viết theo hiểu biết về bài; nếu bạn có tài khoản Premium, hãy **đối chiếu lại đề gốc** (câu chữ, ràng buộc, định dạng ví dụ). Hình minh họa của đề không có trong notebook; ví dụ được mô tả bằng chữ.

Skeleton LeetCode:

```python
"""
# Definition for a Node.
class Node:
    def __init__(self, val=None, next=None):
        self.val = val
        self.next = next
"""

class Solution:
    def insert(self, head: 'Optional[Node]', insertVal: int) -> 'Node':
        ...
```

Ghi chú: bài **có nhiều đáp án đúng** — khi có nhiều chỗ chèn hợp lệ, đề cho phép chọn **bất kỳ** chỗ nào. Vì vậy notebook **không so danh sách đáp án cứng** mà kiểm tra bằng một **hàm kiểm tra tính hợp lệ** (`validate`): vòng vẫn khép kín, vẫn "sắp xếp theo vòng", đủ phần tử cũ và đúng một nút mới, trả về đúng nút ban đầu... Notebook cũng tự dựng lớp `Node` vòng và các hàm dựng/in vòng.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho **một nút bất kỳ** của danh sách liên kết **vòng** đã sắp xếp không giảm (nút đó không nhất thiết là nút nhỏ nhất), chèn `insertVal` sao cho danh sách vẫn "sắp xếp theo vòng"; danh sách rỗng thì tạo vòng một nút; ngược lại trả về **nút ban đầu**.
2. Nắm các công cụ Python: lớp `Node(val, next)`, danh sách vòng (`last.next = first`), duyệt **đúng một vòng** bằng điều kiện dừng `cur is head`, so sánh danh tính bằng `is`, hai con trỏ `prev`/`curr` kề nhau, và khái niệm **"một chỗ tụt" (descent)** của dãy sắp xếp theo vòng.
3. Đi từ **thử mọi chỗ rồi kiểm tra** (`O(n²)`) → **hai con trỏ `prev`/`curr` trong một vòng** → **tìm nút lớn nhất (điểm gãy) rồi chèn như danh sách sắp xếp thường** → **tìm "tiền nhiệm lớn nhất không vượt quá `insertVal`"**.
4. Hiểu mấu chốt: dãy sắp xếp theo vòng có **tối đa một chỗ tụt** (từ lớn nhất về nhỏ nhất); chỗ chèn hợp lệ là (a) khe giữa hai nút `prev ≤ v ≤ curr`, hoặc (b) **ngay tại chỗ tụt** khi `v ≥ lớn nhất` hoặc `v ≤ nhỏ nhất`; nếu không khe nào thỏa thì mọi giá trị đều bằng nhau và chèn ở đâu cũng được.
5. Biết cách **test bài có nhiều đáp án đúng** bằng hàm kiểm tra tính hợp lệ, và tránh các lỗi kinh điển: quên danh sách rỗng, quên trường hợp chèn ở chỗ tụt, vòng lặp vô hạn khi mọi giá trị bằng nhau, nối thiếu `node.next`, trả về nút mới thay vì nút ban đầu.

## 2. Đề bài & ví dụ

Cho một nút `head` của **danh sách liên kết vòng** (nút cuối trỏ về nút đầu) đã **sắp xếp không giảm** theo chiều đi. `head` có thể là **nút bất kỳ** của vòng (không nhất thiết nhỏ nhất). Hãy viết hàm chèn giá trị `insertVal` vào danh sách sao cho danh sách **vẫn là vòng đã sắp xếp**.

- Nếu có **nhiều chỗ** chèn phù hợp, bạn có thể chọn **bất kỳ** chỗ nào.
- Nếu danh sách **rỗng** (`head` là `None`): tạo một vòng mới chỉ có **một nút** (nút trỏ về chính nó) và **trả về nút đó**.
- Ngược lại, **trả về nút `head` ban đầu**.

**Ví dụ 1:** `head = [3, 4, 1]` (vòng 3 → 4 → 1 → 3), `insertVal = 2` → `[3, 4, 1, 2]`. Chèn `2` giữa `1` và `3`; vẫn trả về nút `3`.

**Ví dụ 2:** `head = []`, `insertVal = 1` → `[1]`.

**Ví dụ 3:** `head = [1]`, `insertVal = 0` → `[1, 0]` (vòng 1 → 0 → 1; vòng hai nút `1, 0` sắp xếp theo vòng vì `0` rồi `1` rồi quay lại `0`).

(Ký hiệu `[a, b, c]` là các giá trị đi từ nút `head` theo `next`.)

**Ràng buộc:** số nút trong `[0, 5·10^4]`; `-10^6 <= Node.val, insertVal <= 10^6`.

In [1]:
import itertools
import random
import time
from typing import List, Optional


class Node:
    def __init__(self, val=None, next=None):
        self.val = val
        self.next = next


def build_ring(vals):
    """Dựng vòng từ danh sách giá trị theo chiều next; trả về list các nút (nút 0 là head)."""
    nodes = [Node(v) for v in vals]
    for i, nd in enumerate(nodes):
        nd.next = nodes[(i + 1) % len(nodes)]
    return nodes


def ring_nodes(head, limit=10**6):
    """Các nút theo chiều next, bắt đầu từ head, dừng khi quay lại head (có chốt chặn)."""
    if head is None:
        return []
    out, cur = [head], head.next
    while cur is not head:
        out.append(cur)
        assert len(out) <= limit, "vòng không khép kín về head"
        cur = cur.next
    return out


def ring_vals(head):
    return [nd.val for nd in ring_nodes(head)]


nodes = build_ring([3, 4, 1])
print("vòng:", ring_vals(nodes[0]), "| nút cuối trỏ về head?", nodes[-1].next is nodes[0])
print("bắt đầu từ nút '4':", ring_vals(nodes[1]))
print("ví dụ 3, [1] chèn 0 -> mong đợi (một chỗ chèn hợp lệ):", [1, 0])

vòng: [3, 4, 1] | nút cuối trỏ về head? True
bắt đầu từ nút '4': [4, 1, 3]
ví dụ 3, [1] chèn 0 -> mong đợi (một chỗ chèn hợp lệ): [1, 0]


## 3. Kiến thức nền

### 3.1. Danh sách liên kết vòng

Ở danh sách thường, nút cuối có `next = None`. Ở danh sách **vòng**, nút cuối có `next` trỏ về **nút đầu**, nên từ **bất kỳ** nút nào đi theo `next` ta đều quay về chính nó sau đúng `n` bước. Vì không còn `None`, vòng lặp `while cur:` sẽ **chạy mãi**; ta phải dừng khi **quay lại nút xuất phát**.

In [2]:
nodes = build_ring([5, 6, 7])
start = nodes[1]                       # xuất phát từ nút giữa (6)
cur, seen = start, []
while True:
    seen.append(cur.val)
    cur = cur.next
    if cur is start:                   # dừng khi quay lại nút xuất phát
        break
print(seen)                            # [6, 7, 5]

[6, 7, 5]


### 3.2. So sánh danh tính: `is` và `is not`

Hai nút có thể cùng giá trị nhưng **khác đối tượng**. Điều kiện "đã quay lại đầu vòng" phải so **đối tượng** (`cur is head`), **không** so giá trị (`cur.val == head.val`) — nếu có nhiều nút cùng giá trị, so giá trị sẽ dừng sớm.

In [3]:
nodes = build_ring([2, 2, 2])
head = nodes[0]
print(nodes[1].val == head.val, nodes[1] is head, nodes[2] is head)
steps, cur = 0, head.next
while cur is not head:
    cur = cur.next
    steps += 1
print("đi", steps, "bước để về lại head (n - 1 =", len(nodes) - 1, ")")

True False False
đi 2 bước để về lại head (n - 1 = 2 )


### 3.3. Vòng một nút và vòng rỗng

Vòng **một nút** có `node.next = node` (trỏ về chính nó). Vòng **rỗng** biểu diễn bằng `None` (không có nút nào). Khi `head is None` ta tạo nút mới và nối `node.next = node`.

In [4]:
single = Node(9)
single.next = single
print(single.next is single, ring_vals(single))
print(ring_vals(None))

True [9]
[]


### 3.4. Hai con trỏ kề nhau `prev`, `curr`

Để xét từng **khe** giữa hai nút liên tiếp ta giữ `prev` và `curr = prev.next`, rồi tiến cả hai: `prev, curr = curr, curr.next`. Mỗi vòng ta thấy một khe `(prev, curr)`; một vòng `n` nút có đúng `n` khe (kể cả khe `(nút cuối, nút đầu)`).

In [5]:
nodes = build_ring([1, 3, 5])
head = nodes[0]
prev, curr = head, head.next
gaps = []
while True:
    gaps.append((prev.val, curr.val))
    prev, curr = curr, curr.next
    if prev is head:
        break
print(gaps)             # 3 khe: (1,3), (3,5), (5,1)

[(1, 3), (3, 5), (5, 1)]


### 3.5. "Sắp xếp theo vòng" và **chỗ tụt** (descent)

Dãy các giá trị theo vòng, ví dụ `[3, 4, 1]`, **không** không-giảm theo nghĩa thường, nhưng nó là một dãy sắp xếp `1, 3, 4` **bị quay** đi. Dấu hiệu: đi quanh vòng, giá trị **giảm** (`prev > curr`) **không quá một lần**. Chỗ giảm đó là **chỗ tụt** từ giá trị lớn nhất về nhỏ nhất. Nếu mọi giá trị bằng nhau thì **không có** chỗ tụt.

In [6]:
def descents(vals):
    n = len(vals)
    return sum(1 for i in range(n) if vals[i] > vals[(i + 1) % n])


for vals in ([3, 4, 1], [1, 3, 5], [2, 2, 2], [1, 3, 2], [4, 1, 3, 2], [1]):
    print(f"{str(vals):<14} số chỗ tụt = {descents(vals)} -> sắp xếp theo vòng? {descents(vals) <= 1}")

[3, 4, 1]      số chỗ tụt = 1 -> sắp xếp theo vòng? True
[1, 3, 5]      số chỗ tụt = 1 -> sắp xếp theo vòng? True
[2, 2, 2]      số chỗ tụt = 0 -> sắp xếp theo vòng? True
[1, 3, 2]      số chỗ tụt = 2 -> sắp xếp theo vòng? False
[4, 1, 3, 2]   số chỗ tụt = 2 -> sắp xếp theo vòng? False
[1]            số chỗ tụt = 0 -> sắp xếp theo vòng? True


### 3.6. Bài có nhiều đáp án: kiểm tra bằng hàm hợp lệ

Khi đề cho phép nhiều đáp án đúng, ta **không** so với một đáp án cố định mà viết **hàm kiểm tra** đáp án có thỏa các điều kiện của đề hay không. Ở mục 12 hàm `validate` kiểm tra: vòng khép kín quay về `head`; còn đủ các nút cũ **cùng thứ tự vòng**; có **đúng một** nút mới mang giá trị `insertVal`; số chỗ tụt `≤ 1`; hàm trả về đúng nút theo yêu cầu.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy **Ví dụ 1**: vòng `3 → 4 → 1 → (về 3)`, chèn `2`, cầm nút `3`.

```
      ┌──────────────┐
      ▼              │
     [3] → [4] → [1] ┘        các khe: (3,4), (4,1), (1,3)
```

Sắp xếp thật sự của vòng là `1, 3, 4` (quay vòng). Ta tìm **khe** mà `prev ≤ 2 ≤ curr`:

- khe `(3, 4)`: `3 ≤ 2`? Không.
- khe `(4, 1)`: đây là **chỗ tụt** (4 → 1). `2` nằm giữa `1` và `4` nên **không** thuộc chỗ tụt.
- khe `(1, 3)`: `1 ≤ 2 ≤ 3` ✓ → chèn giữa `1` và `3`.

Kết quả vòng `3 → 4 → 1 → 2 → (về 3)`: từ nút `3` là `[3, 4, 1, 2]` ✓.

Hai tình huống khác để tự thử:

- Chèn **`0`** vào `[3, 4, 1]`: `0` nhỏ hơn mọi giá trị. Không khe thường nào chứa `0`; ta chèn **tại chỗ tụt** `(4, 1)` → `4 → 0 → 1`. Tương tự chèn **`9`** (lớn hơn mọi giá trị) cũng vào chỗ tụt: `4 → 9 → 1`.
- Vòng **`[3, 3, 3]`** chèn `5` hoặc `1`: mọi khe đều `(3, 3)`, không có chỗ tụt; chèn ở đâu cũng hợp lệ.

## 5. Cách 1 — Brute Force / baseline: thử **mọi chỗ**, kiểm tra rồi giữ chỗ hợp lệ đầu tiên

**Ý tưởng:** đi dọc vòng, với mỗi nút `p` thử **chèn nút mới ngay sau `p`**, rồi **kiểm tra cả vòng** có còn sắp xếp theo vòng (số chỗ tụt `≤ 1`) hay không. Hợp lệ thì dừng; không thì **gỡ** nút mới ra và thử `p` tiếp theo. Danh sách rỗng và các thao tác nối con trỏ như thông thường.

Đây là bản dịch trực tiếp của đề ("chèn sao cho vẫn sắp xếp"), nên cũng là **oracle** đáng tin để đối chiếu; ta cũng dùng nó để **liệt kê mọi chỗ chèn hợp lệ**.

In [7]:
def is_sorted_circular(head):
    """True nếu dãy giá trị theo vòng có không quá một chỗ tụt."""
    if head is None:
        return True
    vals = ring_vals(head)
    return descents(vals) <= 1


def valid_gaps(head, v):
    """Liệt kê chỉ số các nút p (theo thứ tự từ head) sao cho chèn v ngay sau p vẫn hợp lệ."""
    res = []
    for idx, p in enumerate(ring_nodes(head)):
        node = Node(v, p.next)
        p.next = node
        ok = is_sorted_circular(head)
        p.next = node.next                      # gỡ nút ra
        if ok:
            res.append(idx)
    return res


def insert_brute(head, insertVal):
    node = Node(insertVal)
    if head is None:
        node.next = node
        return node
    p = head
    while True:
        node.next = p.next
        p.next = node
        if is_sorted_circular(head):
            return head                          # giữ chỗ hợp lệ đầu tiên
        p.next = node.next                       # gỡ ra, thử chỗ khác
        p = p.next
        if p is head:
            raise AssertionError("luôn tồn tại ít nhất một chỗ hợp lệ")


nodes = build_ring([3, 4, 1])
print("các chỗ hợp lệ (chèn 2 sau nút thứ ...):", valid_gaps(nodes[0], 2))
nodes = build_ring([3, 4, 1])
r = insert_brute(nodes[0], 2)
print(ring_vals(r), r is nodes[0])
assert ring_vals(r) == [3, 4, 1, 2] and r is nodes[0]

các chỗ hợp lệ (chèn 2 sau nút thứ ...): [2]
[3, 4, 1, 2] True


**Độ phức tạp:** có `n` chỗ để thử, mỗi lần kiểm tra cả vòng `O(n)` ⇒ **`O(n²)`**; với `n = 5·10^4` là cỡ `2,5 × 10^9` — quá chậm. Space `O(n)` (do `ring_vals` tạo danh sách; có thể kiểm tra `O(1)` bộ nhớ).

**Nút thắt:** ta kiểm tra **cả vòng** cho mỗi chỗ thử, trong khi chèn một nút **chỉ ảnh hưởng một khe**. Chỉ cần xét **khe** `(prev, curr)` quanh chỗ chèn: nó có "hợp lệ" hay không chỉ phụ thuộc `prev.val`, `curr.val`, `insertVal`.

## 6. Ý tưởng thuật toán / bước đột phá

Gọi khe `(prev, curr)` là hai nút kề nhau (theo `next`). Chèn `v` vào giữa khe **giữ nguyên tính hợp lệ** khi:

1. **Khe thường (không tụt, `prev ≤ curr`):** cần `prev ≤ v ≤ curr` — `v` nằm giữa.
2. **Khe tụt (`prev > curr`, đi từ lớn nhất về nhỏ nhất):** cần `v ≥ prev` (không nhỏ hơn giá trị lớn nhất) **hoặc** `v ≤ curr` (không lớn hơn giá trị nhỏ nhất).

**Bổ đề — luôn có khe hợp lệ.** Vòng có tối đa một khe tụt.

- Nếu `min ≤ v ≤ max` và có chỗ tụt: tồn tại khe thường chứa `v` (các nút từ nhỏ nhất đến lớn nhất không giảm), hoặc `v` bằng/bên ngoài một đầu và rơi vào luật 2.
- Nếu `v > max` hoặc `v < min`: luật 2 thỏa ở khe tụt duy nhất.
- Nếu **không có khe tụt**: mọi giá trị bằng nhau (đi một vòng không giảm mà về lại giá trị đầu ⇒ hằng) và **mọi khe đều hợp lệ** (`prev = curr`, chèn `v` bên cạnh chỉ tạo tối đa một chỗ tụt).

Do đó chỉ cần **một vòng** xét từng khe; nếu duyệt hết mà **không khe nào** thỏa luật 1–2 (chỉ xảy ra khi mọi giá trị bằng nhau, hoặc khe cuối chưa được xét) thì chèn ở **khe cuối** `(nút cuối, head)`.

Từ đó có các cách: **hai con trỏ** (Cách 2), **tìm điểm gãy rồi chèn như danh sách sắp xếp** (Cách 3), **tìm tiền nhiệm lớn nhất `≤ v`** (Cách 4).

## 7. Các cách tiếp theo

Tổng cộng 4 cách: Cách 1 (mục 5) và ba cách dưới đây. Cách 2 xét **từng khe** với luật 1–2; Cách 3 **đưa vòng về danh sách thường** bằng cách tìm điểm gãy (nút lớn nhất cuối) rồi chèn như bài sắp xếp quen thuộc; Cách 4 nhìn bài như **tìm tiền nhiệm**: nút có giá trị lớn nhất mà không vượt quá `v`.

### Cách 2 — Hai con trỏ `prev`/`curr` trong đúng một vòng (cách chính thức)

**Giải thích từng đoạn:**

1. Danh sách rỗng: tạo `node`, `node.next = node`, trả về `node`.
2. `prev = head`, `curr = head.next`. Lặp **khi `curr is not head`** (xét các khe `(head, head.next)`, ..., chưa xét khe cuối `(cuối, head)`):
   - Nếu **khe thường chứa `v`** (`prev.val <= v <= curr.val`) hoặc **khe tụt phù hợp** (`prev.val > curr.val` và (`v >= prev.val` hoặc `v <= curr.val`)): **dừng**.
   - Ngược lại tiến: `prev, curr = curr, curr.next`.
3. Chèn `node` vào giữa `prev` và `curr` (khi vòng lặp kết thúc mà không dừng, `prev` là nút cuối và `curr` là `head`: đúng khe cuối).
4. Trả về `head`.

In [8]:
def insert_two_pointers(head, insertVal):
    node = Node(insertVal)
    if head is None:
        node.next = node
        return node
    prev, curr = head, head.next
    while curr is not head:
        if prev.val <= insertVal <= curr.val or (
            prev.val > curr.val and (insertVal >= prev.val or insertVal <= curr.val)
        ):
            break
        prev, curr = curr, curr.next
    prev.next = node
    node.next = curr
    return head


nodes = build_ring([3, 4, 1])
r = insert_two_pointers(nodes[0], 2)
print(ring_vals(r))
assert ring_vals(r) == [3, 4, 1, 2] and r is nodes[0]

[3, 4, 1, 2]


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def insert(self, head: 'Optional[Node]', insertVal: int) -> 'Node':
        node = Node(insertVal)
        if head is None:
            node.next = node
            return node
        prev, curr = head, head.next
        while curr != head:
            if prev.val <= insertVal <= curr.val or (
                prev.val > curr.val and (insertVal >= prev.val or insertVal <= curr.val)
            ):
                break
            prev, curr = curr, curr.next
        prev.next = node
        node.next = curr
        return head
```

**Độ phức tạp:** duyệt tối đa một vòng ⇒ **`O(n)`** thời gian, `O(1)` bộ nhớ phụ.

### Cách 3 — Tìm **điểm gãy** (nút lớn nhất) rồi chèn như danh sách sắp xếp thường

**Giải thích từng đoạn:**

1. Rỗng: như trên.
2. **Tìm điểm gãy** `pivot`: nút `p` có `p.val > p.next.val` — đây là **nút lớn nhất cuối cùng** (chỗ tụt duy nhất); `pivot.next` là nút **nhỏ nhất đầu tiên**. Nếu **không có** điểm gãy (mọi giá trị bằng nhau) thì chèn ngay sau `head`.
3. Có điểm gãy: coi danh sách là dãy **thường** `lo = pivot.next → ... → pivot`:
   - `v >= pivot.val` (không nhỏ hơn lớn nhất) hoặc `v <= lo.val` (không lớn hơn nhỏ nhất): chèn **sau `pivot`** (tức trước `lo`).
   - Ngược lại: bắt đầu `cur = lo`, đi `while cur.next.val < v: cur = cur.next`, rồi chèn sau `cur` (như chèn vào danh sách đơn đã sắp xếp).
4. Trả về `head`.

Đây là cách "bóc vòng thành danh sách" — hữu ích khi bạn muốn tái dùng bài chèn vào danh sách sắp xếp quen thuộc.

In [9]:
def insert_pivot(head, insertVal):
    node = Node(insertVal)
    if head is None:
        node.next = node
        return node

    # 1) tìm điểm gãy: nút lớn nhất cuối cùng (p.val > p.next.val)
    pivot, cur = None, head
    while True:
        if cur.val > cur.next.val:
            pivot = cur
            break
        cur = cur.next
        if cur is head:
            break

    if pivot is None:                                   # mọi giá trị bằng nhau: chèn ở đâu cũng được
        node.next = head.next
        head.next = node
        return head

    lo = pivot.next                                     # nhỏ nhất đầu tiên
    if insertVal >= pivot.val or insertVal <= lo.val:   # ngoài [min, max] hoặc bằng đầu mút
        node.next = lo
        pivot.next = node
        return head

    cur = lo                                            # min < v < max: chèn như danh sách thường
    while cur.next.val < insertVal:
        cur = cur.next
    node.next = cur.next
    cur.next = node
    return head


nodes = build_ring([3, 4, 1])
r = insert_pivot(nodes[0], 2)
print(ring_vals(r))
assert ring_vals(r) == [3, 4, 1, 2] and r is nodes[0]

[3, 4, 1, 2]


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def insert(self, head: 'Optional[Node]', insertVal: int) -> 'Node':
        node = Node(insertVal)
        if head is None:
            node.next = node
            return node
        pivot, cur = None, head
        while True:
            if cur.val > cur.next.val:
                pivot = cur
                break
            cur = cur.next
            if cur is head:
                break
        if pivot is None:
            node.next = head.next
            head.next = node
            return head
        lo = pivot.next
        if insertVal >= pivot.val or insertVal <= lo.val:
            node.next = lo
            pivot.next = node
            return head
        cur = lo
        while cur.next.val < insertVal:
            cur = cur.next
        node.next = cur.next
        cur.next = node
        return head
```

**Độ phức tạp:** một lượt tìm điểm gãy `O(n)` + một lượt chèn `O(n)` ⇒ `O(n)`; `O(1)` bộ nhớ phụ.

### Cách 4 — Tìm **tiền nhiệm lớn nhất không vượt quá `v`** (quét một vòng, ghi nhớ ứng viên)

**Giải thích từng đoạn:**

1. Chèn `v` sau nút `p` là hợp lệ khi `p` là **nút cuối của cụm bằng nhau** có giá trị **lớn nhất trong những giá trị `≤ v`**: khi đó nút kế `p.next` hoặc có giá trị lớn hơn (`> v` hoặc ít nhất `> p.val`) hoặc là nút nhỏ nhất (đã vòng qua chỗ tụt).
2. Quét **một vòng**. Với mỗi nút `cur` (có `nx = cur.next`):
   - ghi nhớ `drop = cur` nếu `nx.val < cur.val` (nút lớn nhất cuối);
   - nếu `cur.val <= v` và `nx.val != cur.val` (`cur` là nút cuối của cụm bằng nhau) thì `best = cur` nếu `best` chưa có hoặc `cur.val > best.val`.
3. Chọn nơi chèn: `best` nếu có; nếu không có giá trị nào `≤ v` (tức `v` nhỏ hơn mọi giá trị) thì chèn sau `drop` (nút lớn nhất, trước nút nhỏ nhất); nếu cũng không có `drop` (mọi giá trị bằng nhau) thì chèn sau `head`.

Cách nghĩ khác Cách 2: **không dựng điều kiện theo khe**, chỉ tìm một **nút đứng trước** phù hợp.

In [10]:
def insert_pred(head, insertVal):
    node = Node(insertVal)
    if head is None:
        node.next = node
        return node

    best = drop = None
    cur = head
    while True:
        nx = cur.next
        if nx.val < cur.val:
            drop = cur                                  # nút lớn nhất cuối cùng
        if cur.val <= insertVal and nx.val != cur.val:  # cur là nút cuối của cụm bằng nhau, và <= v
            if best is None or cur.val > best.val:
                best = cur
        cur = nx
        if cur is head:
            break

    target = best if best is not None else (drop if drop is not None else head)
    node.next = target.next
    target.next = node
    return head


nodes = build_ring([3, 4, 1])
r = insert_pred(nodes[0], 2)
print(ring_vals(r))
assert ring_vals(r) == [3, 4, 1, 2] and r is nodes[0]

[3, 4, 1, 2]


**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def insert(self, head: 'Optional[Node]', insertVal: int) -> 'Node':
        node = Node(insertVal)
        if head is None:
            node.next = node
            return node
        best = drop = None
        cur = head
        while True:
            nx = cur.next
            if nx.val < cur.val:
                drop = cur
            if cur.val <= insertVal and nx.val != cur.val:
                if best is None or cur.val > best.val:
                    best = cur
            cur = nx
            if cur is head:
                break
        target = best if best is not None else (drop if drop is not None else head)
        node.next = target.next
        target.next = node
        return head
```

**Độ phức tạp:** một vòng ⇒ `O(n)`; `O(1)` bộ nhớ phụ.

### Chạy thử ba ví dụ của đề với cả bốn cách

Vì có nhiều chỗ hợp lệ, ta chỉ in kết quả; việc kiểm tra tính hợp lệ nằm ở `validate` (mục 12).

In [11]:
IMPLS = {
    "brute": insert_brute,
    "two_pointers": insert_two_pointers,
    "pivot": insert_pivot,
    "pred": insert_pred,
}
cases = [([3, 4, 1], 2), ([], 1), ([1], 0)]
for vals, v in cases:
    row = {}
    for name, f in IMPLS.items():
        nodes = build_ring(vals) if vals else []
        head = nodes[0] if nodes else None
        r = f(head, v)
        row[name] = ring_vals(r)
    print(f"head={vals}, insertVal={v}:", row)

head=[3, 4, 1], insertVal=2: {'brute': [3, 4, 1, 2], 'two_pointers': [3, 4, 1, 2], 'pivot': [3, 4, 1, 2], 'pred': [3, 4, 1, 2]}
head=[], insertVal=1: {'brute': [1], 'two_pointers': [1], 'pivot': [1], 'pred': [1]}
head=[1], insertVal=0: {'brute': [1, 0], 'two_pointers': [1, 0], 'pivot': [1, 0], 'pred': [1, 0]}


## 8. TRACE từng bước

**Trace Cách 2** trên `head = [3, 4, 1]`, `insertVal = 2` (xuất phát từ nút `3`):

| Vòng | `prev` → `curr` | Khe thường `prev ≤ 2 ≤ curr`? | Khe tụt (`prev > curr`) và (`2 ≥ prev` hoặc `2 ≤ curr`)? | Hành động |
|---|---|---|---|---|
| 1 | `3 → 4` | `3 ≤ 2`? Không | `3 > 4`? Không | tiến |
| 2 | `4 → 1` | `4 ≤ 2`? Không | `4 > 1` ✓, nhưng `2 ≥ 4`? Không; `2 ≤ 1`? Không | tiến |
| — | `1 → 3` (`curr is head`) | (không xét trong vòng lặp) | — | thoát vòng lặp, chèn giữa `1` và `3` |

Sau chèn: `3 → 4 → 1 → 2 → 3`. Hàm `trace_two_pointers` in từng khe:

In [12]:
def trace_two_pointers(vals, insertVal, start=0):
    nodes = build_ring(vals)
    head = nodes[start]
    print(f"vòng từ head: {ring_vals(head)}, insertVal={insertVal}")
    node = Node(insertVal)
    prev, curr = head, head.next
    while curr is not head:
        normal = prev.val <= insertVal <= curr.val
        drop = prev.val > curr.val and (insertVal >= prev.val or insertVal <= curr.val)
        print(f"  khe ({prev.val},{curr.val}): thường={normal!s:<5} tụt phù hợp={drop!s:<5}", end=" -> ")
        if normal or drop:
            print("DỪNG, chèn tại đây")
            break
        print("tiến")
        prev, curr = curr, curr.next
    else:
        print(f"  hết vòng lặp: chèn ở khe cuối ({prev.val},{curr.val})")
    prev.next = node
    node.next = curr
    print("  kết quả:", ring_vals(head), "\n")
    return head


h = trace_two_pointers([3, 4, 1], 2)
assert ring_vals(h) == [3, 4, 1, 2]
h = trace_two_pointers([3, 4, 1], 0)               # nhỏ hơn mọi giá trị: chèn tại chỗ tụt
assert ring_vals(h) == [3, 4, 0, 1]
h = trace_two_pointers([3, 4, 1], 9)               # lớn hơn mọi giá trị: cũng chỗ tụt
assert ring_vals(h) == [3, 4, 9, 1]
h = trace_two_pointers([3, 3, 3], 5)               # mọi giá trị bằng nhau: rơi xuống khe cuối
assert ring_vals(h) == [3, 3, 3, 5]
h = trace_two_pointers([1, 3, 5], 4, start=2)      # head là nút lớn nhất (5): vòng 5 -> 1 -> 3
assert ring_vals(h) == [5, 1, 3, 4]

vòng từ head: [3, 4, 1], insertVal=2
  khe (3,4): thường=False tụt phù hợp=False -> tiến
  khe (4,1): thường=False tụt phù hợp=False -> tiến
  hết vòng lặp: chèn ở khe cuối (1,3)
  kết quả: [3, 4, 1, 2] 

vòng từ head: [3, 4, 1], insertVal=0
  khe (3,4): thường=False tụt phù hợp=False -> tiến
  khe (4,1): thường=False tụt phù hợp=True  -> DỪNG, chèn tại đây
  kết quả: [3, 4, 0, 1] 

vòng từ head: [3, 4, 1], insertVal=9
  khe (3,4): thường=False tụt phù hợp=False -> tiến
  khe (4,1): thường=False tụt phù hợp=True  -> DỪNG, chèn tại đây
  kết quả: [3, 4, 9, 1] 

vòng từ head: [3, 3, 3], insertVal=5
  khe (3,3): thường=False tụt phù hợp=False -> tiến
  khe (3,3): thường=False tụt phù hợp=False -> tiến
  hết vòng lặp: chèn ở khe cuối (3,3)
  kết quả: [3, 3, 3, 5] 

vòng từ head: [5, 1, 3], insertVal=4
  khe (5,1): thường=False tụt phù hợp=False -> tiến
  khe (1,3): thường=False tụt phù hợp=False -> tiến
  hết vòng lặp: chèn ở khe cuối (3,5)
  kết quả: [5, 1, 3, 4] 



**Trace Cách 3 (điểm gãy):** in vòng, điểm gãy `pivot`, `lo` rồi quyết định.

In [13]:
def trace_pivot(vals, insertVal, start=0):
    head = build_ring(vals)[start]
    print(f"vòng từ head: {ring_vals(head)}, insertVal={insertVal}")
    pivot, cur = None, head
    while True:
        if cur.val > cur.next.val:
            pivot = cur
            break
        cur = cur.next
        if cur is head:
            break
    if pivot is None:
        print("  không có điểm gãy -> mọi giá trị bằng nhau -> chèn sau head")
    else:
        lo = pivot.next
        print(f"  pivot (lớn nhất cuối) = {pivot.val}, lo (nhỏ nhất đầu) = {lo.val}")
        if insertVal >= pivot.val or insertVal <= lo.val:
            print(f"  {insertVal} >= {pivot.val} hoặc <= {lo.val} -> chèn sau pivot")
        else:
            c = lo
            path = [c.val]
            while c.next.val < insertVal:
                c = c.next
                path.append(c.val)
            print(f"  đi từ lo qua {path}, chèn sau {c.val}")
    r = insert_pivot(head, insertVal)
    print("  kết quả:", ring_vals(r), "\n")
    return r


assert ring_vals(trace_pivot([3, 4, 1], 2)) == [3, 4, 1, 2]
assert ring_vals(trace_pivot([1, 3, 5, 7], 6, start=2)) == [5, 6, 7, 1, 3]
assert ring_vals(trace_pivot([2, 2, 2], 9)) == [2, 9, 2, 2]

vòng từ head: [3, 4, 1], insertVal=2
  pivot (lớn nhất cuối) = 4, lo (nhỏ nhất đầu) = 1
  đi từ lo qua [1], chèn sau 1
  kết quả: [3, 4, 1, 2] 

vòng từ head: [5, 7, 1, 3], insertVal=6
  pivot (lớn nhất cuối) = 7, lo (nhỏ nhất đầu) = 1
  đi từ lo qua [1, 3, 5], chèn sau 5
  kết quả: [5, 6, 7, 1, 3] 

vòng từ head: [2, 2, 2], insertVal=9
  không có điểm gãy -> mọi giá trị bằng nhau -> chèn sau head
  kết quả: [2, 9, 2, 2] 



**Trace Cách 4 (tiền nhiệm):** với mỗi nút in `best` và `drop` hiện tại.

In [14]:
def trace_pred(vals, insertVal, start=0):
    head = build_ring(vals)[start]
    print(f"vòng từ head: {ring_vals(head)}, insertVal={insertVal}")
    best = drop = None
    cur = head
    while True:
        nx = cur.next
        if nx.val < cur.val:
            drop = cur
        if cur.val <= insertVal and nx.val != cur.val:
            if best is None or cur.val > best.val:
                best = cur
        print(f"  cur={cur.val} nx={nx.val}  best={best.val if best else None}  drop={drop.val if drop else None}")
        cur = nx
        if cur is head:
            break
    target = best if best is not None else (drop if drop is not None else head)
    print(f"  chèn sau nút {target.val}")
    r = insert_pred(build_ring(vals)[start], insertVal)
    print("  kết quả:", ring_vals(r), "\n")
    return r


assert ring_vals(trace_pred([3, 4, 1], 2)) == [3, 4, 1, 2]
assert ring_vals(trace_pred([3, 4, 1], 0)) == [3, 4, 0, 1]

vòng từ head: [3, 4, 1], insertVal=2
  cur=3 nx=4  best=None  drop=None
  cur=4 nx=1  best=None  drop=4
  cur=1 nx=3  best=1  drop=4
  chèn sau nút 1
  kết quả: [3, 4, 1, 2] 

vòng từ head: [3, 4, 1], insertVal=0
  cur=3 nx=4  best=None  drop=None
  cur=4 nx=1  best=None  drop=4
  cur=1 nx=3  best=None  drop=4
  chèn sau nút 4
  kết quả: [3, 4, 0, 1] 



## 9. Code nộp LeetCode

Ta nộp **Cách 2** (hai con trỏ trong một vòng): ngắn, `O(n)`, `O(1)` bộ nhớ, rất phổ biến.

In [15]:
class Solution:
    def insert(self, head: 'Optional[Node]', insertVal: int) -> 'Node':
        node = Node(insertVal)
        if head is None:
            node.next = node
            return node
        prev, curr = head, head.next
        while curr != head:
            if prev.val <= insertVal <= curr.val or (
                prev.val > curr.val and (insertVal >= prev.val or insertVal <= curr.val)
            ):
                break
            prev, curr = curr, curr.next
        prev.next = node
        node.next = curr
        return head


nodes = build_ring([3, 4, 1])
r = Solution().insert(nodes[0], 2)
print(ring_vals(r))
assert ring_vals(r) == [3, 4, 1, 2] and r is nodes[0]
IMPLS["Solution"] = lambda head, v: Solution().insert(head, v)

[3, 4, 1, 2]


## 10. Vì sao đúng

**Điều kiện hợp lệ.** Dãy giá trị theo vòng là "sắp xếp theo vòng" khi số chỗ tụt (`prev > curr` trên một khe) `≤ 1`. Chèn `v` vào khe `(prev, curr)` thay khe đó bằng hai khe `(prev, v)` và `(v, curr)`; các khe khác giữ nguyên. Số chỗ tụt mới = số chỗ tụt cũ − `[prev > curr]` + `[prev > v]` + `[v > curr]`.

- **Khe thường** (`prev ≤ curr`) và `prev ≤ v ≤ curr`: `[prev > v] = 0`, `[v > curr] = 0` ⇒ số chỗ tụt **không đổi**.
- **Khe tụt** (`prev > curr`) và `v ≥ prev`: `[prev > v] = 0`, `[v > curr] = 1` (vì `v ≥ prev > curr`) ⇒ số chỗ tụt: `−1 + 0 + 1 = 0` thay đổi ⇒ vẫn **1**. Với `v ≤ curr`: `[prev > v] = 1` (vì `v ≤ curr < prev`), `[v > curr] = 0` ⇒ vẫn 1.
- **Mọi giá trị bằng nhau**, khe `(c, c)` không tụt: chèn `v` sinh `[c > v] + [v > c]` = đúng 1 (nếu `v ≠ c`) hoặc 0 (nếu `v = c`) ⇒ số chỗ tụt `≤ 1` ✓.

**Luôn dừng ở một khe hợp lệ (Cách 2):** vòng lặp xét các khe `(head, ·)`, ..., `(·, cuối)`; chỉ khe cuối `(cuối, head)` không được xét trong vòng lặp. Nếu không khe nào dừng, ta chèn đúng vào khe cuối. Ta chứng minh khe cuối là hợp lệ: theo mục 6 mọi vòng đều có ít nhất một khe hợp lệ; các khe đã xét đều không thỏa ⇒ khe còn lại phải thỏa (hoặc vòng gồm toàn giá trị bằng nhau, nơi mọi khe đều hợp lệ).

**Cách 3:** `pivot` là nút lớn nhất cuối (duy nhất nếu có tụt); `pivot.next = lo` là nhỏ nhất đầu. Từ `lo` đến `pivot` là một dãy **không giảm**; `v ≥ pivot.val` hoặc `v ≤ lo.val` ⇒ chèn tại chỗ tụt (hợp lệ theo luật khe tụt). Còn `lo.val < v < pivot.val` ⇒ tồn tại `cur` trong dãy với `cur.val < v ≤ cur.next.val`; vòng `while cur.next.val < v` dừng ở đó (vì `pivot.val > v` nên không vượt `pivot`) ⇒ khe thường hợp lệ.

**Cách 4:** `best` là nút cuối cùng của cụm bằng nhau có giá trị lớn nhất `≤ v`: nút kế của nó có giá trị khác `best.val` — hoặc lớn hơn (`> v` do `best` là cực đại `≤ v` ... hoặc là cụm giá trị `> v`), hoặc nhỏ hơn (khi `best` là cụm lớn nhất: chỗ tụt). Cả hai trường hợp đều khe hợp lệ. Nếu không có giá trị nào `≤ v`, `v` nhỏ hơn mọi giá trị ⇒ chỗ tụt (`drop`). Nếu không có tụt, mọi giá trị bằng nhau ⇒ chèn đâu cũng được.

**Trả về:** rỗng ⇒ nút mới tự trỏ chính nó; ngược lại `head` không bị đổi.

Mục 12 kiểm chứng mọi điều trên bằng `validate` trên toàn bộ vòng nhỏ và hàng nghìn vòng ngẫu nhiên.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số nút (`≤ 5·10^4`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử mọi chỗ + kiểm tra cả vòng | `O(n²)` | `O(n)` (hoặc `O(1)`) | oracle, dùng liệt kê mọi chỗ hợp lệ |
| 2. Hai con trỏ | `O(n)` | `O(1)` | **nộp bài** |
| 3. Điểm gãy + chèn thường | `O(n)` (hai lượt) | `O(1)` | dễ giải thích; tái dùng "chèn vào danh sách sắp xếp" |
| 4. Tiền nhiệm lớn nhất `≤ v` | `O(n)` (một lượt) | `O(1)` | không cần dựng điều kiện theo khe |

Mọi cách tối ưu đều `O(n)` vì cần **ít nhất** một lần quét để tìm khe phù hợp (danh sách liên kết không truy cập ngẫu nhiên).

## 12. Bộ test

Vì có **nhiều đáp án đúng**, ta kiểm tra bằng hàm **`validate`** thay vì so đáp án cứng. Với vòng gốc, ta chụp `snapshot` (các nút theo thứ tự vòng từ `head`), chạy hàm, rồi `validate` kiểm:

1. **Vòng khép kín:** từ `head` theo `next` quay lại `head` sau đúng `n + 1` bước (không đứt, không vòng phụ).
2. **Giữ nguyên các nút cũ:** các nút cũ vẫn là **cùng đối tượng** và giữ **thứ tự vòng** như trước.
3. **Đúng một nút mới**, giá trị bằng `insertVal`.
4. **Sắp xếp theo vòng:** số chỗ tụt `≤ 1`.
5. **Giá trị trả về:** nút `head` ban đầu (vòng không rỗng); với vòng rỗng, một nút duy nhất tự trỏ về chính nó, giá trị `insertVal`.

Rồi: ví dụ đề, edge case (rỗng, một nút, mọi giá trị bằng nhau, `head` ở giữa vòng, giá trị nhỏ nhất/lớn nhất, trùng giá trị), **vét cạn** các vòng nhỏ, **ngẫu nhiên có seed**, **vòng lớn** `5·10^4`, và **thời gian**.

In [16]:
def snapshot(head):
    nodes = ring_nodes(head)
    return {"nodes": nodes, "vals": [n.val for n in nodes], "head": head}


def validate(snap, v, result):
    """Trả về None nếu hợp lệ, ngược lại là chuỗi mô tả lỗi."""
    old = snap["nodes"]
    if not old:
        if result is None:
            return "vòng rỗng phải trả về một nút mới"
        if result.next is not result:
            return "nút duy nhất phải tự trỏ về chính nó"
        if result.val != v:
            return "giá trị nút mới sai"
        return None
    head = snap["head"]
    if result is not head:
        return "phải trả về nút head ban đầu"
    n = len(old)
    seq, cur = [head], head.next
    while cur is not head:
        seq.append(cur)
        if len(seq) > n + 1 or cur is None:
            return "vòng không khép kín về head (đứt hoặc có vòng phụ)"
        cur = cur.next
        if cur is None:
            return "vòng bị đứt (gặp None)"
    if len(seq) != n + 1:
        return f"số nút sau khi chèn là {len(seq)}, đáng lẽ {n + 1}"
    old_ids = {id(x) for x in old}
    new = [x for x in seq if id(x) not in old_ids]
    if len(new) != 1 or new[0].val != v:
        return "phải có đúng một nút mới mang giá trị insertVal"
    kept = [x for x in seq if x is not new[0]]
    if kept != old:
        return "các nút cũ không giữ nguyên thứ tự vòng"
    if descents([x.val for x in seq]) > 1:
        return "vòng không còn sắp xếp theo vòng (nhiều hơn một chỗ tụt)"
    return None


def run_case(f, vals, v, start=0):
    nodes = build_ring(vals) if vals else []
    head = nodes[start] if nodes else None
    snap = snapshot(head)
    result = f(head, v)
    return validate(snap, v, result)


# 1) Ví dụ đề
for name, f in IMPLS.items():
    for vals, v in [([3, 4, 1], 2), ([], 1), ([1], 0)]:
        assert run_case(f, vals, v) is None, (name, vals, v)
print("Ví dụ đề (4 cách + Solution): validate OK")

# kiểm tra chính validate: nó phải BẮT được lời giải sai
nodes = build_ring([3, 4, 1])
snap = snapshot(nodes[0])
nodes[2].next = Node(0, nodes[0])                  # chèn 0 giữa 1 và 3 -> 3,4,1,0: hai chỗ tụt? (4>1, 1>0)
assert validate(snap, 0, nodes[0]) is not None
print("validate bắt được chèn sai:", validate(snap, 0, nodes[0]))

Ví dụ đề (4 cách + Solution): validate OK
validate bắt được chèn sai: vòng không còn sắp xếp theo vòng (nhiều hơn một chỗ tụt)


In [17]:
# 2) Edge case
edge = [
    ([], 0), ([], -10**6), ([], 10**6),                         # rỗng
    ([5], 5), ([5], 4), ([5], 6), ([-10**6], 10**6),            # một nút
    ([2, 2, 2, 2], 2), ([2, 2, 2, 2], 1), ([2, 2, 2, 2], 3),    # mọi giá trị bằng nhau
    ([1, 2], 0), ([1, 2], 3), ([1, 2], 1), ([1, 2], 2),
    ([1, 3, 3, 5], 3), ([1, 3, 3, 5], 4), ([1, 3, 3, 5], 0), ([1, 3, 3, 5], 9),
    ([1, 1, 3], 1), ([1, 3, 3], 3), ([1, 1, 3], 0), ([1, 1, 3], 5),
    ([-5, -5, 0, 7, 7], 7), ([-5, -5, 0, 7, 7], -5), ([-5, -5, 0, 7, 7], 8), ([-5, -5, 0, 7, 7], -6),
]
count = 0
for vals, v in edge:
    starts = range(len(vals)) if vals else [0]
    for st in starts:                                           # head có thể là BẤT KỲ nút nào
        for name, f in IMPLS.items():
            err = run_case(f, vals, v, st)
            assert err is None, (name, vals, v, st, err)
            count += 1
print(f"Edge case: OK ({len(edge)} trường hợp, mọi vị trí head, {count} lượt chạy)")

Edge case: OK (26 trường hợp, mọi vị trí head, 375 lượt chạy)


In [18]:
# 3) Vét cạn: mọi vòng sắp xếp theo vòng với giá trị trong {0,1,2,3}, kích thước <= 5, mọi head, mọi insertVal trong -1..4
total = 0
multi = 0                                                       # số trường hợp có NHIỀU chỗ chèn hợp lệ
for n in range(1, 6):
    for sorted_vals in itertools.combinations_with_replacement(range(4), n):
        for rot in range(n):                                    # quay: vòng bắt đầu từ vị trí rot của dãy đã sắp xếp
            ring = list(sorted_vals[rot:] + sorted_vals[:rot])
            for st in range(n):
                for v in range(-1, 5):
                    for name, f in IMPLS.items():
                        err = run_case(f, ring, v, st)
                        assert err is None, (name, ring, v, st, err)
                    total += 1
                    if len(valid_gaps(build_ring(ring)[st], v)) > 1:
                        multi += 1
print(f"Vét cạn: {total} cặp (vòng, head, insertVal) x 5 cài đặt OK; {multi} cặp có nhiều chỗ chèn hợp lệ")
assert multi > 1000

Vét cạn: 13104 cặp (vòng, head, insertVal) x 5 cài đặt OK; 6284 cặp có nhiều chỗ chèn hợp lệ


In [19]:
# 3b) Minh họa "nhiều đáp án đúng": các cài đặt khác nhau có thể chèn ở chỗ khác nhau, đều hợp lệ
ring, v = [1, 3, 3, 5], 3
gaps = valid_gaps(build_ring(ring)[0], v)
print("vòng", ring, "chèn", v, "-> các chỗ hợp lệ (chèn sau nút thứ ...):", gaps)
for name, f in IMPLS.items():
    nodes = build_ring(ring)
    r = f(nodes[0], v)
    where = [i for i, x in enumerate(ring_nodes(r)) if x.val == v and x not in nodes]   # vị trí nút mới
    print(f"  {name:<13} -> {ring_vals(r)}  (nút mới ở vị trí {where})")

vòng [1, 3, 3, 5] chèn 3 -> các chỗ hợp lệ (chèn sau nút thứ ...): [0, 1, 2]
  brute         -> [1, 3, 3, 3, 5]  (nút mới ở vị trí [1])
  two_pointers  -> [1, 3, 3, 3, 5]  (nút mới ở vị trí [1])
  pivot         -> [1, 3, 3, 3, 5]  (nút mới ở vị trí [1])
  pred          -> [1, 3, 3, 3, 5]  (nút mới ở vị trí [3])
  Solution      -> [1, 3, 3, 3, 5]  (nút mới ở vị trí [1])


In [20]:
# 4) Ngẫu nhiên có seed: vòng sắp xếp theo vòng (có trùng), head bất kỳ, insertVal bất kỳ
def gen_ring(rng):
    n = rng.choice([0, 1, 2, 3, 5, 8, 15, 30])
    kind = rng.random()
    if kind < 0.15:
        vals = [rng.randint(-5, 5)] * n                          # mọi giá trị bằng nhau
    else:
        span = rng.choice([3, 10, 10**6])
        vals = sorted(rng.randint(-span, span) for _ in range(n))
    rot = rng.randrange(n) if n else 0
    ring = vals[rot:] + vals[:rot]
    st = rng.randrange(n) if n else 0
    span = max(3, max(map(abs, vals), default=3)) + 2
    v = rng.randint(-span, span) if rng.random() < 0.7 else rng.choice(vals + [-10**6, 10**6] if vals else [0])
    return ring, v, st


rng = random.Random(708)
cnt = 0
for _ in range(5000):
    ring, v, st = gen_ring(rng)
    assert all(-10**6 <= x <= 10**6 for x in ring) and -10**6 <= v <= 10**6
    for name, f in IMPLS.items():
        err = run_case(f, ring, v, st)
        assert err is None, (name, ring, v, st, err)
    cnt += 1
print(f"Ngẫu nhiên: {cnt} test x 5 cài đặt: validate OK")

Ngẫu nhiên: 5000 test x 5 cài đặt: validate OK


In [21]:
# 5) Vòng lớn 5*10^4: dùng validate (O(n)) chỉ cho các cách nhanh; brute chỉ chạy trên vòng nhỏ
N = 50000
rng = random.Random(5)
vals = sorted(rng.randint(-10**6, 10**6) for _ in range(N))
rot = 12345
ring = vals[rot:] + vals[:rot]
for v in (-10**6, 10**6, 0, vals[100], vals[-1], vals[0], 777):
    for st in (0, 20000, N - 1):
        for name in ("two_pointers", "pivot", "pred", "Solution"):
            err = run_case(IMPLS[name], ring, v, st)
            assert err is None, (name, v, st, err)
print(f"Vòng {N} nút: 3 head x 7 insertVal x 4 cách validate OK")
allsame = [7] * N
for v in (7, 0, 9):
    for name in ("two_pointers", "pivot", "pred", "Solution"):
        assert run_case(IMPLS[name], allsame, v, 0) is None
print("Vòng 5*10^4 nút toàn giá trị bằng nhau: OK")

Vòng 50000 nút: 3 head x 7 insertVal x 4 cách validate OK


Vòng 5*10^4 nút toàn giá trị bằng nhau: OK


In [22]:
# 6) Thời gian
def timeit(f, ring, v, st, reps=3):
    best = 1e18
    for _ in range(reps):
        nodes = build_ring(ring)
        t0 = time.perf_counter()
        f(nodes[st], v)
        best = min(best, (time.perf_counter() - t0) * 1000)
    return best


vals = list(range(2000))
ring = vals[700:] + vals[:700]
print("n = 2000 (insertVal = -1: phải đi hết phần lớn vòng):")
for name, f in IMPLS.items():
    if name == "Solution":
        continue
    print(f"  {name:<13} {timeit(f, ring, -1, 1500, reps=1 if name == 'brute' else 3):9.3f} ms")
vals = list(range(50000))
ring = vals[25000:] + vals[:25000]
print("n = 50000 (brute bị bỏ qua vì O(n^2)):")
for name in ("two_pointers", "pivot", "pred"):
    print(f"  {name:<13} {timeit(IMPLS[name], ring, 49999, 0):9.3f} ms")

n = 2000 (insertVal = -1: phải đi hết phần lớn vòng):


  brute           458.565 ms
  two_pointers      0.105 ms
  pivot             0.078 ms
  pred              0.110 ms
n = 50000 (brute bị bỏ qua vì O(n^2)):
  two_pointers      2.097 ms
  pivot             0.998 ms


  pred              5.359 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1: không xử lý danh sách **rỗng**

`head` là `None` mà viết `head.next` sẽ báo `AttributeError`. Phải tạo nút mới, cho `node.next = node` và trả về nút đó.

In [23]:
def wrong_no_empty(head, insertVal):
    node = Node(insertVal)
    prev, curr = head, head.next                  # head = None -> AttributeError
    return head


try:
    wrong_no_empty(None, 1)
except AttributeError as e:
    print("AttributeError:", e)
assert run_case(insert_two_pointers, [], 1) is None
print("cách đúng: validate ->", run_case(insert_two_pointers, [], 1))

AttributeError: 'NoneType' object has no attribute 'next'
cách đúng: validate -> None


### Lỗi 2: chỉ xét khe thường, quên **chỗ tụt** (chèn giá trị lớn nhất/nhỏ nhất)

Chỉ kiểm tra `prev.val <= insertVal <= curr.val`: với `insertVal` nhỏ hơn mọi giá trị hoặc lớn hơn mọi giá trị, không khe nào thỏa nên lời giải rơi xuống chèn ở khe cuối — **sai chỗ**, phá thứ tự.

In [24]:
def wrong_no_wrap(head, insertVal):
    node = Node(insertVal)
    if head is None:
        node.next = node
        return node
    prev, curr = head, head.next
    while curr is not head:
        if prev.val <= insertVal <= curr.val:           # sai: thiếu trường hợp chỗ tụt
            break
        prev, curr = curr, curr.next
    prev.next = node
    node.next = curr
    return head


print("[3,4,1] chèn 2:", run_case(wrong_no_wrap, [3, 4, 1], 2))          # tình cờ đúng (rơi vào khe cuối hợp lệ)
print("[3,4,1] chèn 0:", run_case(wrong_no_wrap, [3, 4, 1], 0))
print("[3,4,1] chèn 9:", run_case(wrong_no_wrap, [3, 4, 1], 9))
assert run_case(wrong_no_wrap, [3, 4, 1], 0) is not None and run_case(insert_two_pointers, [3, 4, 1], 0) is None

[3,4,1] chèn 2: None
[3,4,1] chèn 0: vòng không còn sắp xếp theo vòng (nhiều hơn một chỗ tụt)
[3,4,1] chèn 9: vòng không còn sắp xếp theo vòng (nhiều hơn một chỗ tụt)


### Lỗi 3: `while True` không có chốt chặn ⇒ **vòng lặp vô hạn** khi mọi giá trị bằng nhau

Nếu chỉ lặp `while not (khe thỏa)` mà không có điều kiện "đã đi hết một vòng", thì với vòng `[3, 3, 3]` và `insertVal = 5` không khe nào thỏa (mọi khe `(3, 3)`), nên vòng lặp chạy mãi. Dưới đây ta đặt **chốt chặn** để quan sát lỗi thay vì treo máy.

In [25]:
def wrong_infinite(head, insertVal, max_steps=1000):
    node = Node(insertVal)
    if head is None:
        node.next = node
        return node
    prev, curr = head, head.next
    steps = 0
    while not (prev.val <= insertVal <= curr.val or
               (prev.val > curr.val and (insertVal >= prev.val or insertVal <= curr.val))):
        prev, curr = curr, curr.next                     # sai: không dừng khi đã đi hết một vòng
        steps += 1
        if steps > max_steps:
            raise RuntimeError(f"đã đi {steps} bước mà vẫn chưa dừng: vòng lặp vô hạn")
    prev.next = node
    node.next = curr
    return head


try:
    run_case(wrong_infinite, [3, 3, 3], 5)
except RuntimeError as e:
    print("RuntimeError:", e)
assert run_case(wrong_infinite, [3, 4, 1], 2) is None            # với vòng bình thường vẫn chạy đúng
assert run_case(insert_two_pointers, [3, 3, 3], 5) is None

RuntimeError: đã đi 1001 bước mà vẫn chưa dừng: vòng lặp vô hạn


### Lỗi 4: nối con trỏ thiếu `node.next = curr` ⇒ vòng bị **đứt**

Chỉ viết `prev.next = node` mà quên cho `node.next` trỏ tiếp; nút mới có `next = None`, vòng đứt, các nút phía sau bị mất.

In [26]:
def wrong_broken_link(head, insertVal):
    node = Node(insertVal)
    if head is None:
        node.next = node
        return node
    prev, curr = head, head.next
    while curr is not head:
        if prev.val <= insertVal <= curr.val or (
            prev.val > curr.val and (insertVal >= prev.val or insertVal <= curr.val)
        ):
            break
        prev, curr = curr, curr.next
    prev.next = node                                    # sai: quên node.next = curr
    return head


print(run_case(wrong_broken_link, [3, 4, 1], 2))
assert run_case(wrong_broken_link, [3, 4, 1], 2) is not None

vòng bị đứt (gặp None)


### Lỗi 5: trả về **nút mới** thay vì nút `head` ban đầu (với vòng không rỗng)

Đề yêu cầu trả về `head` ban đầu khi vòng không rỗng. Trả về nút mới làm đầu vào của lệnh gọi kế tiếp (hoặc trình chấm) thay đổi.

In [27]:
def wrong_return_new(head, insertVal):
    node = Node(insertVal)
    if head is None:
        node.next = node
        return node
    prev, curr = head, head.next
    while curr is not head:
        if prev.val <= insertVal <= curr.val or (
            prev.val > curr.val and (insertVal >= prev.val or insertVal <= curr.val)
        ):
            break
        prev, curr = curr, curr.next
    prev.next = node
    node.next = curr
    return node                                         # sai: phải trả về head


print(run_case(wrong_return_new, [3, 4, 1], 2))
assert run_case(wrong_return_new, [3, 4, 1], 2) == "phải trả về nút head ban đầu"

phải trả về nút head ban đầu


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Nhận dạng:** "danh sách vòng đã sắp xếp, `head` là **nút bất kỳ**" ⇒ nhớ có **một chỗ tụt** (lớn nhất → nhỏ nhất). Mọi bài về dãy **quay** (Search in Rotated Sorted Array, Find Minimum in Rotated Sorted Array) đều xoay quanh chỗ tụt này.
- **Ba tình huống cần xét:** khe thường (`prev ≤ v ≤ curr`), khe tụt (`v ≥ max` hoặc `v ≤ min`), và **mọi giá trị bằng nhau**. Liệt kê trước khi code.
- **Duyệt vòng:** điều kiện dừng là `cur is head` (so **danh tính**), tuyệt đối không `while cur:`; thêm chốt chặn khi gỡ lỗi.
- **Trường hợp đặc biệt:** `head is None`; vòng một nút (`next` là chính nó).
- **Giá trị trùng:** dùng `<=`/`>=` ở các biên khe; chèn bên cạnh cụm bằng nhau là hợp lệ.
- **Bài nhiều đáp án:** viết **`validate`** (khép kín, giữ nút cũ, đúng một nút mới, số chỗ tụt `≤ 1`) và test với vét cạn nhỏ + ngẫu nhiên.
- **Cùng họ:** *Insert into a Sorted List* (kiểu 147 Insertion Sort List), *Linked List Cycle* (141), *Rotate List* (61), *Circular Array Loop* (457), *Design Circular Queue* (622).

## 15. Biến thể của bài

1. **Danh sách vòng không giảm → không tăng** (sắp xếp giảm dần): đổi chiều so sánh.
2. **Xóa một giá trị** khỏi vòng đã sắp xếp (và trả về nút hợp lệ nếu `head` bị xóa).
3. **Chèn vào danh sách thường đã sắp xếp** (*Insertion Sort List*, LeetCode 147): cùng ý tưởng nhưng không có chỗ tụt.
4. **Đếm số chỗ tụt / kiểm tra một vòng có sắp xếp theo vòng không** (bài tập 5).
5. **Chèn nhiều giá trị** một lần: sắp xếp các giá trị rồi gộp (merge) một lượt.
6. **Chèn khi có thể là danh sách chưa sắp xếp** (không bảo đảm): không tồn tại đáp án tổng quát — cần xác thực đầu vào trước.

Cài nhanh biến thể 2 (xóa giá trị đầu tiên gặp) và kiểm tra tính hợp lệ:

In [28]:
def delete_value(head, val):
    """Xóa nút đầu tiên (theo chiều từ head) có giá trị val; trả về nút head mới (None nếu rỗng)."""
    if head is None:
        return None
    prev, cur = head, head.next
    if head.val == val:                                # xóa chính head
        if head.next is head:
            return None
        last = head
        while last.next is not head:
            last = last.next
        last.next = head.next
        return head.next
    while cur is not head:
        if cur.val == val:
            prev.next = cur.next
            break
        prev, cur = cur, cur.next
    return head


rng = random.Random(15)
for _ in range(1000):
    ring, v, st = gen_ring(rng)
    if not ring:
        continue
    nodes = build_ring(ring)
    head = nodes[st]
    before = ring_vals(head)
    new_head = delete_value(head, v)
    if v in before:
        idx = before.index(v)
        expect = before[:idx] + before[idx + 1:]
        got = ring_vals(new_head) if new_head else []
        if idx == 0:
            assert sorted(got) == sorted(expect) and len(got) == len(expect)
        else:
            assert got == expect
    else:
        assert ring_vals(new_head) == before
    assert is_sorted_circular(new_head)
print("Biến thể xóa giá trị: sau khi xóa vòng vẫn sắp xếp theo vòng: OK")

Biến thể xóa giá trị: sau khi xóa vòng vẫn sắp xếp theo vòng: OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi chỗ + kiểm tra | dịch nguyên văn đề | `O(n²)` | `O(n)` | oracle, liệt kê chỗ hợp lệ |
| 2. Hai con trỏ theo khe | luật khe thường / khe tụt, dừng khi thỏa | `O(n)` | `O(1)` | **làm bài** |
| 3. Điểm gãy + chèn thường | bóc vòng thành danh sách thường | `O(n)` | `O(1)` | khi muốn tái dùng "chèn vào danh sách sắp xếp" |
| 4. Tiền nhiệm lớn nhất `≤ v` | quét một vòng, nhớ ứng viên | `O(n)` | `O(1)` | không muốn dựng điều kiện khe; dễ mở rộng sang xóa/tìm |

Trực giác chọn: **hai con trỏ** là mặc định. Nếu bạn quen tư duy "dãy quay quanh điểm gãy" thì **Cách 3**. Cách 1 chỉ để kiểm chứng.

## 17. 5 bài tập vận dụng (dễ → khó)

Mọi cell TODO **không gọi** hàm khi chạy notebook; đáp án có assert ở mục 19.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

```python
nodes = build_ring([1, 3, 5])
r = insert_two_pointers(nodes[0], 4)
print(ring_vals(r))
nodes = build_ring([2, 2, 2])
r = insert_two_pointers(nodes[0], 5)
print(ring_vals(r))
```

In [29]:
nodes_b1 = build_ring([1, 3, 5])
r = insert_two_pointers(nodes_b1[0], 4)
print(ring_vals(r))
nodes_b1b = build_ring([2, 2, 2])
r = insert_two_pointers(nodes_b1b[0], 5)
print(ring_vals(r))

[1, 3, 4, 5]
[2, 2, 2, 5]


### Bài 2 (tính tay)

Vòng `5 → 1 → 2 → 4 → (về 5)`, `head` là nút `5`, `insertVal = 3`. Theo **Cách 2** (hai con trỏ), vòng lặp dừng ở khe nào và danh sách (từ `head`) cuối cùng là gì? Ghi vào `my_answer_2` (list; để `None` đến khi tính xong).

In [30]:
ring_b2, v_b2 = [5, 1, 2, 4], 3
my_answer_2 = None          # TODO: điền list các giá trị từ head sau khi chèn, ví dụ [5, 1, 2, 4, 3]

### Bài 3 (điền TODO)

Hoàn thành điều kiện dừng của hàm `insert_todo` (hai nhánh: khe thường, khe tụt). Cell chỉ **định nghĩa**, không gọi.

In [31]:
def insert_todo(head, insertVal):
    node = Node(insertVal)
    if head is None:
        node.next = node
        return node
    prev, curr = head, head.next
    while curr is not head:
        # TODO: nếu (khe thường chứa insertVal) hoặc (khe tụt phù hợp) thì break
        prev, curr = curr, curr.next
    prev.next = node
    node.next = curr
    return head

### Bài 4 (tự viết lại)

Tự viết `insert_mine(head, insertVal)` theo **Cách 3** (tìm điểm gãy `pivot`, rồi chèn như danh sách sắp xếp thường). Không nhìn code cũ. Cell dưới chỉ để khung rỗng.

In [32]:
def insert_mine(head, insertVal):
    # TODO: viết theo Cách 3
    pass

### Bài 5 (biến thể / mini challenge)

Viết `is_sorted_circular_mine(head)`: trả về `True` nếu vòng bắt đầu từ `head` là **sắp xếp theo vòng** (số chỗ tụt `≤ 1`), duyệt đúng một vòng bằng con trỏ (không dựng danh sách phụ), `head = None` cho `True`.

In [33]:
def is_sorted_circular_mine(head):
    # TODO: đếm số chỗ tụt (cur.val > cur.next.val) trong đúng một vòng
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: vòng đầu có khe `(3, 5)` chứa `4`. Vòng thứ hai: mọi khe là `(2, 2)`.
- Bài 2: lần lượt xét khe `(5,1)`, `(1,2)`, `(2,4)`... dừng khi thỏa.
- Bài 3: viết hai điều kiện nối bằng `or`.
- Bài 4: tìm nút `p` có `p.val > p.next.val`.
- Bài 5: mỗi lần `cur.val > cur.next.val` tăng một biến đếm.

**Mức 2 (vừa):**
- Bài 1: chèn `4` giữa `3` và `5`. Với `[2, 2, 2]` chèn `5`: không khe nào thỏa, rơi xuống khe cuối.
- Bài 2: `(5,1)` là khe tụt: `3 ≥ 5`? không; `3 ≤ 1`? không → tiến. `(1,2)`: `1 ≤ 3 ≤ 2`? không. `(2,4)`: `2 ≤ 3 ≤ 4` ✓.
- Bài 3: khe thường `prev.val <= insertVal <= curr.val`; khe tụt `prev.val > curr.val and (insertVal >= prev.val or insertVal <= curr.val)`.
- Bài 4: nếu không có điểm gãy thì mọi giá trị bằng nhau; có thì `lo = pivot.next`.
- Bài 5: vòng lặp dừng khi `cur.next is head`; nhớ xét cả khe `(cuối, head)`.

**Mức 3 (gần đáp án):**
- Bài 1: `[1, 3, 4, 5]` và `[2, 2, 2, 5]`.
- Bài 2: chèn giữa `2` và `4`: `[5, 1, 2, 3, 4]`.
- Bài 3: `if prev.val <= insertVal <= curr.val or (prev.val > curr.val and (insertVal >= prev.val or insertVal <= curr.val)): break`.
- Bài 4: `if insertVal >= pivot.val or insertVal <= lo.val:` chèn sau `pivot`; ngược lại `while cur.next.val < insertVal: cur = cur.next` bắt đầu từ `lo`.
- Bài 5: `cur = head; while True: if cur.val > cur.next.val: cnt += 1; cur = cur.next; if cur is head: break; return cnt <= 1`.

## 19. Đáp án tham khảo

Phần này đặt cuối bài giảng; mọi đáp án có assert.

In [34]:
# Bài 1
assert ring_vals(r) == [2, 2, 2, 5]
assert ring_vals(nodes_b1[0]) == [1, 3, 4, 5]
print("Bài 1: [1, 3, 4, 5] và [2, 2, 2, 5]")

# Bài 2
ans_2 = [5, 1, 2, 3, 4]
nodes = build_ring(ring_b2)
r2 = insert_two_pointers(nodes[0], v_b2)
assert ring_vals(r2) == ans_2
assert run_case(insert_two_pointers, ring_b2, v_b2) is None
print("Bài 2:", ans_2, "(dừng ở khe (2, 4))")


# Bài 3
def insert_todo_sol(head, insertVal):
    node = Node(insertVal)
    if head is None:
        node.next = node
        return node
    prev, curr = head, head.next
    while curr is not head:
        if prev.val <= insertVal <= curr.val or (
            prev.val > curr.val and (insertVal >= prev.val or insertVal <= curr.val)
        ):
            break
        prev, curr = curr, curr.next
    prev.next = node
    node.next = curr
    return head


rng = random.Random(3)
for _ in range(500):
    ring, v, st = gen_ring(rng)
    assert run_case(insert_todo_sol, ring, v, st) is None
print("Bài 3: OK")

Bài 1: [1, 3, 4, 5] và [2, 2, 2, 5]
Bài 2: [5, 1, 2, 3, 4] (dừng ở khe (2, 4))
Bài 3: OK


In [35]:
# Bài 4
def insert_mine_sol(head, insertVal):
    node = Node(insertVal)
    if head is None:
        node.next = node
        return node
    pivot, cur = None, head
    while True:
        if cur.val > cur.next.val:
            pivot = cur
            break
        cur = cur.next
        if cur is head:
            break
    if pivot is None:
        node.next = head.next
        head.next = node
        return head
    lo = pivot.next
    if insertVal >= pivot.val or insertVal <= lo.val:
        node.next = lo
        pivot.next = node
        return head
    cur = lo
    while cur.next.val < insertVal:
        cur = cur.next
    node.next = cur.next
    cur.next = node
    return head


rng = random.Random(4)
for _ in range(1000):
    ring, v, st = gen_ring(rng)
    assert run_case(insert_mine_sol, ring, v, st) is None
print("Bài 4: OK")


# Bài 5
def is_sorted_circular_mine_sol(head):
    if head is None:
        return True
    cnt, cur = 0, head
    while True:
        if cur.val > cur.next.val:
            cnt += 1
        cur = cur.next
        if cur is head:
            break
    return cnt <= 1


rng = random.Random(5)
for _ in range(1000):
    n = rng.randint(1, 7)
    vals = [rng.randint(0, 4) for _ in range(n)]                  # vòng bất kỳ, nhiều cái không hợp lệ
    head = build_ring(vals)[rng.randrange(n)]
    assert is_sorted_circular_mine_sol(head) == is_sorted_circular(head) == (descents(ring_vals(head)) <= 1)
assert is_sorted_circular_mine_sol(None) is True
assert is_sorted_circular_mine_sol(build_ring([3, 4, 1])[0]) is True
assert is_sorted_circular_mine_sol(build_ring([1, 3, 2])[0]) is False
print("Bài 5: OK")

Bài 4: OK
Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (hai con trỏ trong một vòng):**

```python
class Solution:
    def insert(self, head, insertVal):
        node = Node(insertVal)
        if head is None:                       # vòng rỗng
            node.next = node
            return node
        prev, curr = head, head.next
        while curr is not head:                # đúng một vòng (chưa xét khe cuối)
            if prev.val <= insertVal <= curr.val or (
                prev.val > curr.val and (insertVal >= prev.val or insertVal <= curr.val)
            ):
                break                          # khe thường, hoặc chỗ tụt
            prev, curr = curr, curr.next
        prev.next = node                       # (không khe nào: khe cuối, hoặc mọi giá trị bằng nhau)
        node.next = curr
        return head                            # trả về head ban đầu
```

**Công thức nhớ:** *khe thường `prev ≤ v ≤ curr`; khe tụt `prev > curr` và (`v ≥ prev` hoặc `v ≤ curr`); không khe nào ⇒ chèn ở khe cuối.*

**Checklist trước khi submit:**

- [ ] `head is None` ⇒ tạo nút tự trỏ về chính nó, trả về nút mới.
- [ ] Duyệt vòng bằng `curr is not head` (so danh tính), không `while curr:`; không lặp vô hạn khi mọi giá trị bằng nhau.
- [ ] Có **cả hai** trường hợp: khe thường và **khe tụt** (giá trị lớn nhất/nhỏ nhất).
- [ ] Nối **đủ** `prev.next = node` và `node.next = curr`.
- [ ] Trả về **`head` ban đầu** (vòng không rỗng).
- [ ] Đã test: rỗng, một nút, mọi giá trị bằng nhau, `head` ở giữa vòng, trùng giá trị, giá trị nhỏ nhất/lớn nhất; test bằng hàm `validate` (nhiều đáp án đúng).
- [ ] Bài Premium: đã đối chiếu đề gốc nếu có tài khoản.